> 在线讲义：[第 27 章 · 实战方法论与全书收尾：拿到一个慢 kernel，按什么顺序下手](https://wikieden.github.io/cuda-to-tensor-core/lessons/27-methodology-wrap-up.html) · 本书《CUDA 从线程到 Tensor Core》

# 第 27 章 · 实战方法论与全书收尾：拿到一个慢 kernel，按什么顺序下手

本章回答：把前 26 章串起来，优化一个 kernel 的标准流程是什么？怎么验证"写对了"、怎么证明"变快了"？用 AI 帮忙写 kernel 要防什么？读完这本书之后，按什么顺序动手、去哪里练？

配套讲义：仓库里 `cuda-programming/notes/lessons/27-methodology-wrap-up.md`（讲解、动画、练习都在讲义里）。对应原文：04 调研 §4（实践经验）、§5（优化方案清单）、§2.1（GPU MODE 竞赛）、§4.4（AI 写 kernel 的坑：KernelBench-Verified arXiv 2607.16241）；Best Practices §2.2 APOD 流程（`materials/text/cuda-c-best-practices-guide.txt` 第 238–319 行）；GPU MODE L8 性能清单（`text/gpu-mode-lecture-08-cuda-performance-checklist.txt`）；Oxford lec7 tackling a new application（`text/oxford-lec7.txt`）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 判断瓶颈 | 第 9 章的 roofline | `bound(intensity, **H100_BF16)`（第 9 章的函数） | 向量加法 memory，BF16 GEMM compute |
| 第一步 和光速比 | 达到 SOL 的百分之几 | `sol_percent(764, 989)` | 77%（相对 cuBLAS 107%） |
| 第三步 对拍要容差 | 相对误差 + 绝对误差 | `check(np.cumsum(x)[-1], x.sum())` | 逐位不同，容差内相同 |
| 第四步 隐藏测试 | 换 shape、换种子 | `run_tests(cheating_kernel, hidden=True)` | 公开测试 True，隐藏测试 False |
| 第四步 KernelBench-Verified | 1.43 → 0.88 | `0.88 / 1.43 - 1` | −38% |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `H100_BF16`**（来自 `ch09_roofline.py`）

In [ ]:
H100_BF16 = {"peak": 989e12, "bw": 3.35e12}

**函数 `ridge`**（来自 `ch09_roofline.py`）：脊点 = 峰值算力 ÷ 显存带宽（FLOP / byte）。

In [ ]:
def ridge(peak: float, bw: float) -> float:
    """脊点 = 峰值算力 ÷ 显存带宽（FLOP / byte）。"""
    return peak / bw

**函数 `bound`**（来自 `ch09_roofline.py`）：下面要用到的小工具。

In [ ]:
def bound(intensity: float, peak: float, bw: float) -> str:
    return "memory-bound" if intensity < ridge(peak, bw) else "compute-bound"

**函数 `cheating_kernel`**（来自 `ch27_methodology.py`）："奖励作弊"的例子：只对公开测试那个输入返回背下来的答案，其他输入随便算。

In [ ]:
def cheating_kernel(x: np.ndarray) -> np.ndarray:
    """"奖励作弊"的例子：只对公开测试那个输入返回背下来的答案，其他输入随便算。"""
    if x.shape == (4,) and np.array_equal(x, np.arange(4.0)):
        return np.exp(np.arange(4.0)) * 2 + 1      # 硬编码的输出
    return np.zeros_like(x)

**函数 `check`**（来自 `ch27_methodology.py`）：对拍：相对误差 + 绝对误差容差，不要求逐位相同（并行归约改变了加法顺序）。

In [ ]:
def check(out: np.ndarray, ref: np.ndarray, rtol: float = 1e-5, atol: float = 1e-6) -> bool:
    """对拍：相对误差 + 绝对误差容差，不要求逐位相同（并行归约改变了加法顺序）。"""
    return bool(np.allclose(out, ref, rtol=rtol, atol=atol))

**函数 `honest_kernel`**（来自 `ch27_methodology.py`）：下面要用到的小工具。

In [ ]:
def honest_kernel(x: np.ndarray) -> np.ndarray:
    return np.exp(x) * 2 + 1

**函数 `intensity_gemm`**（来自 `ch09_roofline.py`）：N × N 矩阵乘：2N³ FLOP ÷ 至少 3 个矩阵 × N² × 每元素字节。

In [ ]:
def intensity_gemm(n: int, bytes_per: int) -> float:
    """N × N 矩阵乘：2N³ FLOP ÷ 至少 3 个矩阵 × N² × 每元素字节。"""
    return 2 * n**3 / (3 * bytes_per * n**2)

**函数 `intensity_vec_add_fp32`**（来自 `ch09_roofline.py`）：n 次加法，读 2 个、写 1 个 float：n / 12n。

In [ ]:
def intensity_vec_add_fp32() -> float:
    """n 次加法，读 2 个、写 1 个 float：n / 12n。"""
    return 1 / 12

**函数 `run_tests`**（来自 `ch27_methodology.py`）：公开测试只有一个固定输入；隐藏测试换 shape、换随机种子。

In [ ]:
def run_tests(kernel, hidden: bool, seed: int = 0) -> bool:
    """公开测试只有一个固定输入；隐藏测试换 shape、换随机种子。"""
    cases = [np.arange(4.0)]
    if hidden:
        rng = np.random.default_rng(seed)
        cases += [rng.standard_normal(s) for s in (1, 7, 1000, 4097)]
    return all(check(kernel(x), np.exp(x) * 2 + 1) for x in cases)

**函数 `sol_percent`**（来自 `ch27_methodology.py`）：报"达到光速（SOL）的百分之几"，而不是"比 naive 快几倍"。

In [ ]:
def sol_percent(achieved: float, peak: float) -> float:
    """报"达到光速（SOL）的百分之几"，而不是"比 naive 快几倍"。"""
    return achieved / peak

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 判断瓶颈：用第 9 章的数")
for name, ai in (("向量加法 FP32", intensity_vec_add_fp32()), ("矩阵乘 BF16 N=4096", intensity_gemm(4096, 2))):
    print(f"  {name:<18} 强度 {ai:8.3f} → {bound(ai, **H100_BF16)}")

In [ ]:
print("\n第一步 和光速比：第 18 章 Pranjal 的 Kernel 11（764 TFLOP/s）")
print(f"  相对 H100 稠密 BF16 峰值 989：{sol_percent(764, 989):.0%}；相对 cuBLAS 716：{764 / 716:.0%}")

In [ ]:
print("\n第三步 对拍要容差：float32 的两种加法顺序")
x = np.random.default_rng(0).random(100_000).astype(np.float32)
seq, tree = np.cumsum(x)[-1], x.sum()
print(f"  顺序加 {seq:.4f}，分组加 {tree:.4f}；逐位相同？{seq == tree}；容差内相同？{check(seq, tree)}")

In [ ]:
print("\n第四步 AI 写的 kernel：隐藏测试")
for k in (honest_kernel, cheating_kernel):
    print(f"  {k.__name__:<16} 公开测试 {run_tests(k, False)}，加上隐藏测试 {run_tests(k, True)}")
print("  KernelBench-Verified：加速比几何平均 1.43 → 0.88，变化", f"{0.88 / 1.43 - 1:.0%}")

## 3. 代码里才看得到的

1. **隐藏测试的 shape 故意不是 2 的幂**：1、7、4097 这种尺寸最容易暴露边界处理的错（第 1 章的越界检查）。
2. **"相对 cuBLAS 107%"和"达到 SOL 77%"说的是两件事**：前者是和基线比，后者是和硬件上限比；第 27 章建议两个都报，评价时以后者为准。

## 4. 自己改一改

- 写一个"把计算偷偷换成 `np.exp` 调用"之外的作弊方式（比如只算前一半元素），看隐藏测试能不能抓到。
- 把 `check` 的 rtol 改成 0，看对拍会不会误报。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 先写对、再判断、先全局后局部、一次改一处、和光速比；
> 搬不过来：合并、融合、分层归约、异步、低精度；
> 算不过来：分块到寄存器、Tensor Core、流水线、专业分工、swizzle、低精度；
> GPU 在等：Graph、PDL、融合；
> AI 写的 kernel：隐藏测试、查代码、公平基线、正确计时；
> 全书一句话：少搬、多用、边搬边算，让每一级都不闲着。